# 08: Online Evaluation

This notebook demonstrates the **Phase 3 online evaluation suite** — metrics that
run in production against live agent responses and push scores to Langfuse
immediately via `create_score()`.

## Offline vs Online Evaluation

| | Offline (Notebook 07) | Online (this notebook) |
|---|---|---|
| **When it runs** | After agent execution, in batch | In the request path or immediately after |
| **Ground truth** | Dataset labels (answer, rubrics) | None — all from instrumentation |
| **LLM calls** | Yes (quality graders) | Optional (coherence only, async) |
| **Latency impact** | None | Near-zero (counters + one async LLM) |

## What You'll Learn

1. Verification compliance — does every `google_search` get a `web_fetch`?
2. Tool call volume — how many tools did the agent use?
3. Replanning rate — from the `replan_count` counter on `AgentResponse`
4. Retry rate — API rate-limit retries and context overflow resets
5. Response coherence — async LLM judge (does not block the response path)
6. Follow-up question rate — session-level implicit feedback
7. Wiring all metrics in a production-style run

## Prerequisites

Complete Notebooks 01–07. Same `.env` credentials required.

In [ ]:
import asyncio
import os
from pathlib import Path

from aieng.agent_evals.async_client_manager import AsyncClientManager
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.evaluation.online import (
    check_follow_up,
    check_verification_compliance,
    evaluate_coherence_async,
    report_all_online_metrics,
    report_replanning_metrics,
    report_retry_metrics,
    report_tool_call_metrics,
    report_verification_compliance,
)
from aieng.agent_evals.knowledge_qa.notebook import run_with_display
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

## 1. Run the Agent

All online metrics derive from the `AgentResponse` object. We create a Langfuse
trace manually so we have a `trace_id` to attach scores to.

In [ ]:
dataset = DeepSearchQADataset()
example = dataset.get_by_category("Finance & Economics")[0]

# Create a Langfuse trace to receive online scores
trace = langfuse_client.trace(
    name="online-eval-demo",
    input=example.problem,
    metadata={"category": example.problem_category, "answer_type": example.answer_type},
)
trace_id = trace.id
console.print(f"[dim]Langfuse trace ID: {trace_id}[/dim]")

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await run_with_display(agent, example.problem)

trace.update(output=response.text)
console.print(
    Panel(
        f"Duration: {response.total_duration_ms / 1000:.1f}s\n"
        f"Tool calls: {len(response.tool_calls)}\n"
        f"Replan count: {response.replan_count}\n"
        f"Rate limit retries: {response.rate_limit_retries}\n"
        f"Context overflow resets: {response.overflow_resets}",
        title="AgentResponse Summary",
        border_style="blue",
    )
)

## 2. Verification Compliance

Every `google_search` must be followed by a `web_fetch` before the next search.
`vertex_search` calls are exempt — grounding is built in.

This is a **hard rule** — violations mean the agent answered from search snippets
rather than verified source content.

In [ ]:
compliant, violations = check_verification_compliance(response.tool_calls)

status = "[green]COMPLIANT[/green]" if compliant else f"[red]VIOLATIONS at positions: {violations}[/red]"
console.print(f"Verification compliance: {status}")

# Push to Langfuse
report_verification_compliance(response, trace_id, langfuse_client)
console.print("[dim]Score pushed to Langfuse: Online/VerificationCompliance[/dim]")

## 3. Tool Call Volume

Instrumentation-only — counts total tool calls and unique tool types.
Helps detect agents that over-use tools or get stuck in loops.

In [ ]:
from collections import Counter

tool_counts = Counter(tc.get("name", "unknown") for tc in response.tool_calls)

t = Table(title=f"Tool Call Volume ({len(response.tool_calls)} total)")
t.add_column("Tool", style="cyan")
t.add_column("Count", style="white", justify="right")
for tool, count in tool_counts.most_common():
    t.add_row(tool, str(count))
console.print(t)

report_tool_call_metrics(response, trace_id, langfuse_client)
console.print("[dim]Scores pushed: ToolCalls/Total, ToolCalls/UniqueTools[/dim]")

## 4. Replanning Rate (Online)

The agent increments `_replan_count` each time it encounters `/*REPLANNING*/` in the
event stream. This is the online version — pushed per-response to Langfuse.

Unlike the offline replanning evaluator, this version has no threshold check and does
not require a dataset label — it's pure instrumentation.

In [ ]:
plan_steps = len(response.plan.steps) if response.plan else 1
replan_ratio = response.replan_count / max(plan_steps, 1)

console.print(
    f"Replan count: [bold]{response.replan_count}[/bold]  "
    f"Plan steps: {plan_steps}  "
    f"Ratio: {replan_ratio:.3f}"
)

report_replanning_metrics(response, trace_id, langfuse_client)
console.print("[dim]Scores pushed: Online/ReplanCount, Online/ReplanRatio[/dim]")

## 5. Retry Rate

Tracks two types of retries accumulated during `answer_async()`:

- **Rate limit retries** — tenacity retries triggered by `is_retryable_api_error()`
- **Context overflow resets** — session resets triggered by `is_context_overflow_error()`

High retry rates may indicate model capacity issues or context window problems.

In [ ]:
console.print(
    f"Rate limit retries: [bold]{response.rate_limit_retries}[/bold]  "
    f"Context overflow resets: [bold]{response.overflow_resets}[/bold]"
)

report_retry_metrics(response, trace_id, langfuse_client)
console.print("[dim]Scores pushed: Retries/RateLimitRetries, Retries/ContextOverflowResets[/dim]")

## 6. Response Coherence (Async LLM Judge)

An LLM judge that checks whether the ANSWER, SOURCES, and REASONING sections
of the response are internally consistent.

This runs **asynchronously after** the response is delivered — it never blocks
the user-facing response path. In production, use `asyncio.create_task()` to
fire-and-forget.

In [ ]:
# Run directly here (in production this would be asyncio.create_task(...))
await evaluate_coherence_async(
    response_text=response.text,
    trace_id=trace_id,
    langfuse_client=langfuse_client,
)
console.print("[dim]Score pushed: Online/Coherence[/dim]")

## 7. Follow-up Question Rate

Tracks whether users immediately send a clarification request after receiving
a response. A high follow-up rate within 60 seconds is an implicit signal that
the agent's answer was unclear or incomplete.

This is a **session-level** metric — not per-response. It requires the application
to call `check_follow_up()` when the next user message arrives.

In [ ]:
# Simulated example: a clarification arrives 15 seconds after the response
simulated_next_message = "Can you clarify what you mean by that?"
simulated_elapsed_ms = 15_000  # 15 seconds

check_follow_up(
    time_since_response_ms=simulated_elapsed_ms,
    next_message=simulated_next_message,
    trace_id=trace_id,
    langfuse_client=langfuse_client,
    threshold_ms=60_000,
)
console.print("[dim]Follow-up detected and scored: UserFeedback/FollowUpRequired[/dim]")

# Simulated example: a new question arrives 90 seconds later (not a follow-up)
check_follow_up(
    time_since_response_ms=90_000,
    next_message="What is the GDP of Canada?",
    trace_id=trace_id,
    langfuse_client=langfuse_client,
)
console.print("[dim]No follow-up recorded (elapsed > threshold)[/dim]")

## 8. Wiring All Metrics in a Production Run

In production, call `report_all_online_metrics()` after `answer_async()` returns.
It pushes all synchronous metrics and schedules coherence as a background task.

In [ ]:
async def production_answer(question: str) -> str:
    """Production-style handler showing where online metrics are wired."""
    trace = langfuse_client.trace(name="production-run", input=question)
    trace_id = trace.id

    agent = KnowledgeGroundedAgent(enable_planning=True)
    response = await agent.answer_async(question)

    trace.update(output=response.text)

    # Push all online metrics — coherence runs as a background task
    report_all_online_metrics(
        response,
        trace_id=trace_id,
        langfuse_client=langfuse_client,
        run_coherence=True,
    )

    return response.text


# Demo with a quick question
answer = await production_answer(example.problem)
console.print(Panel(
    f"[bold]Answer (first 300 chars):[/bold]\n{answer[:300]}...",
    title="Production Run Complete",
    border_style="green",
))
console.print("[dim]All online metrics pushed to Langfuse:[/dim]")
for metric in [
    "ToolCalls/Total", "ToolCalls/UniqueTools",
    "Retries/RateLimitRetries", "Retries/ContextOverflowResets",
    "Online/ReplanCount", "Online/ReplanRatio",
    "Online/VerificationCompliance",
    "Online/Coherence (async)",
]:
    console.print(f"  • {metric}")

## Summary

In this notebook you:

1. **Ran** the agent and created a Langfuse trace to receive online scores
2. **Checked verification compliance** — deterministic rule, no LLM needed
3. **Measured tool call volume** — instrumentation only
4. **Tracked replanning rate** — from `AgentResponse.replan_count` counter
5. **Tracked retry rate** — rate-limit retries + context overflow resets
6. **Evaluated coherence** — async LLM judge that doesn't block the response
7. **Recorded follow-up rate** — session-level implicit feedback signal
8. **Wired everything** into a production-style `report_all_online_metrics()` call

### Production Checklist

- Call `report_all_online_metrics()` after every `answer_async()` in your serving layer
- Pass `run_coherence=True` only if an asyncio event loop is running (it always is in async serving)
- Call `check_follow_up()` in your message handler when the next user turn arrives
- Monitor `Online/VerificationCompliance` — a sustained drop indicates a system prompt regression